In [38]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score

In [5]:
df = pd.read_csv("C:/Users/Welcome/Desktop/Data_SciencePw/Spam-Detection/artifact/10_11_2026_16_28_40/data_ingestion/train.csv")
df.head()

,Label,Message
0,ham,U call me alter at 11 ok.
1,ham,Subject: re : meter 0980438 - acock production...
2,ham,"Subject: enron / hpl actuals for december 11 ,..."
3,spam,FreeMsg:Feelin kinda lnly hope u like 2 keep m...
4,ham,"Subject: enron actuals for july 13 , 2000\r\nt..."


In [6]:
df.shape

(8129, 2)

In [7]:
df.Label.value_counts()

Label
ham     6437
spam    1692
Name: count, dtype: int64

In [8]:
y = df['Label'].map({"ham": 0, "spam": 1})
texts = df["Message"]
y.dtype

dtype('int64')

In [9]:
y.value_counts()

Label
0    6437
1    1692
Name: count, dtype: int64

In [10]:
y.isna().sum()

0

In [13]:
lemmatizer = nltk.stem.WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [18]:
def clean_text(text):
    text = text.lower()
    words = text.split()
    no_stopword = [w for w in words if w not in stop_words]
    lemmatized = [lemmatizer.lemmatize(w) for w in no_stopword]
    return " ".join(lemmatized)


In [19]:
cleaned = texts.apply(clean_text)

In [22]:
sentence = "Subject: enron actuals for july 13 ..."
print(clean_text(sentence))

subject: enron actuals july 13 ...


In [23]:
sentence = "u call me alter at 11 ok."
print(clean_text(sentence))

u call alter 11 ok.


In [24]:
len(cleaned)

8129

In [25]:
len(y)

8129

In [27]:
pipe = Pipeline([
    ("vectorizer" , CountVectorizer()),
    ("model", MultinomialNB()),
    
])

In [30]:
#Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle= True, random_state=42)
y_pred = cross_val_predict(pipe, cleaned, y, cv = skf)

In [31]:
print(accuracy_score(y, y_pred))
print(confusion_matrix(y, y_pred))
print(classification_report(y, y_pred))

0.9477180465001845
[[6134  303]
 [ 122 1570]]
              precision    recall  f1-score   support

           0       0.98      0.95      0.97      6437
           1       0.84      0.93      0.88      1692

    accuracy                           0.95      8129
   macro avg       0.91      0.94      0.92      8129
weighted avg       0.95      0.95      0.95      8129



In [36]:
results = []
def evaluate(name, pipe, texts, y):
    y_predict = cross_val_predict(pipe, texts, y, cv= skf)
    tn, fp, fn, tp = confusion_matrix(y, y_predict).ravel() #.ravel() is used to convert matrix into single line
    row = {"Name": name,
           "accuracy": round(accuracy_score(y, y_predict),4),
           "precision": round(precision_score(y, y_predict), 4),
           "recall": round(recall_score(y, y_predict), 4),
           "FP" : fp,
           "FN": fn} 
    results.append(row)
    return row



In [37]:
evaluate("baseline: lemma + count + MNB", pipe, cleaned, y)
pd.DataFrame(results)

,Name,accuracy,precision,recall,FP,FN
0,baseline: lemma + count + MNB,0.9477,0.8382,0.9279,303,122


In [42]:
pipe2 = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("model", MultinomialNB(alpha= 0.1))
])

evaluate("lemma + tfidf + MNB", pipe2, cleaned, y)
pd.DataFrame(results)

,Name,accuracy,precision,recall,FP,FN
0,baseline: lemma + count + MNB,0.9477,0.8382,0.9279,303,122
1,lemma + tfidf + MNB,0.8906,0.9975,0.4758,2,887
2,lemma + tfidf + MNB,0.9659,0.9652,0.8676,53,224


In [44]:
alpha_values = [0.01, 0.03, 0.1, 0.3, 1, 3, 5]
vectorizers = {
    "tfidf": TfidfVectorizer(),
     "count":  CountVectorizer()
}
results = []
for vec_name, vec in vectorizers.items():
    for a in alpha_values:
        pipe = Pipeline([
            ("vectorizer", vec),
            ("model", MultinomialNB(alpha= a))
        ])
        evaluate(f"lemma + {vec_name} + MNB(alpha = {a})", pipe, cleaned, y)

pd.DataFrame(results)


,Name,accuracy,precision,recall,FP,FN
0,lemma + tfidf + MNB(alpha = 0.01),0.9648,0.9170,0.9137,140,146
1,lemma + tfidf + MNB(alpha = 0.03),0.9678,0.9360,0.9072,105,157
2,lemma + tfidf + MNB(alpha = 0.1),0.9659,0.9652,0.8676,53,224
3,lemma + tfidf + MNB(alpha = 0.3),0.9442,0.9889,0.7400,14,440
4,lemma + tfidf + MNB(alpha = 1),0.8906,0.9975,0.4758,2,887
5,lemma + tfidf + MNB(alpha = 3),0.8179,1.0000,0.1253,0,1480
6,lemma + tfidf + MNB(alpha = 5),0.7967,1.0000,0.0230,0,1653
7,lemma + count + MNB(alpha = 0.01),0.9365,0.7905,0.9456,424,92
8,lemma + count + MNB(alpha = 0.03),0.9381,0.7945,0.9480,415,88
9,lemma + count + MNB(alpha = 0.1),0.9398,0.8000,0.9480,401,88


We will use MultinomialNB model as a result of my experiment with mentor and from above experiment i reached a conclusion of using TFIDF vectorizer instead of using Countvectorizer as it is giving best response good accuracy as well as at same FP it is giving FN half of FP.